# Issue #3: Standardize parking labels

Inventory facility, category, and subtype values across the shared QPASS loader output. Preserve source values, apply formatting and supported aliases, and flag uncertain labels.


## 1. Load QPASS

Use Dylan’s Issue #1 loader to read the supported source sheets across workbook formats. Workbooks belong in `data/raw/QPASS/`; the loader searches that folder directly. Its audit metadata records skipped sheets and dropped template/summary rows. See `docs/qpass_schema_audit.md` for scope and known issues.


In [1]:
# 1A — Load the supported QPASS workbooks
from pathlib import Path
import importlib
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.qpass_loader import load_all_qpass
from src.labels import prepare_label_records, format_label, format_labels
import src.labels as label_tools

qpass_dir = project_root / "data" / "raw" / "QPASS"
qpass = load_all_qpass(qpass_dir)
loader_audit = qpass.attrs.copy()
print(f"Loaded {len(qpass):,} records from {len(loader_audit.get('files', []))} workbooks.")


/opt/anaconda3/envs/myenv/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/myenv/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/myenv/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/myenv/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/envs/myenv/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will 

Loaded 25,026 records from 16 workbooks.


## 2. Preserve raw labels and review loader coverage

### 2A — Prepare the master label dataframe

Keep the loader's complete table, source fields, and audit metadata. Copy `location`, `type`, and `sub_type` into the required raw columns. Older locations are reconstructed by the loader; original cells remain in `raw_structure` and `raw_lot`. Missing subtype may mean an absent source field or an empty cell; do not infer restrictions or automatically mark every missing subtype unresolved.


In [2]:
# 2A — Preserve raw labels and all loader provenance
# Team exclusion documented in docs/qpass_schema_audit.md.
# Keep the loaded table intact and report excluded records explicitly.
excluded_workbook_names = {"QPASS 2025-2026-3 Winter.xlsm"}
excluded_mask = qpass["source_file"].isin(excluded_workbook_names)
excluded_workbooks_report = (
    qpass.loc[excluded_mask].groupby("source_file").size().rename("records").reset_index()
)
if not excluded_workbooks_report.empty:
    print("Excluded from Issue #3 counts under the documented team decision:")
    display(excluded_workbooks_report)

master_df = prepare_label_records(qpass.loc[~excluded_mask].copy())
master_df.attrs["analysis_excluded_workbooks"] = excluded_workbooks_report.to_dict("records")
print(f"Issue #3 scope: {len(master_df):,} records from {master_df['source_file'].nunique()} workbooks.")
records = master_df
coverage_report = (
    records.groupby(["source_file", "academic_year", "quarter"], dropna=False)
    .agg(records=("source_row", "size"), source_sheets=("sheet_name", "nunique"))
    .reset_index()
)
display(coverage_report.head(20))
print("Audit details remain in loader_audit: files, skipped_sheets, dropped_rows, non_numeric_values.")
# Inspect one file's exclusions when needed, without printing every sheet:
# loader_audit["skipped_sheets"].get("QPASS 2024-2025-2 Fall.xlsm")
# loader_audit["dropped_rows"].get("QPASS 2024-2025-2 Fall.xlsm")
display(records[["facility_raw", "category_raw", "subtype_raw", "source_file", "source_row"]].head(5))


Excluded from Issue #3 counts under the documented team decision:


,source_file,records
0,QPASS 2025-2026-3 Winter.xlsm,1598


Issue #3 scope: 23,428 records from 15 workbooks.


,source_file,academic_year,quarter,records,source_sheets
0,QPASS 2018-2019-1 Summer.xlsx,2018-2019,Summer,2655,4
1,QPASS 2018-2019-2 Fall.xlsx,2018-2019,Fall,2696,4
2,QPASS 2018-2019-3 Winter.xlsx,2018-2019,Winter,1440,2
3,QPASS 2018-2019-4 Spring.xlsx,2018-2019,Spring,1060,2
4,QPASS 2019-2020-1 Summer.xlsx,2019-2020,Summer,1120,2
5,QPASS 2019-2020-2 Fall.xlsx,2019-2020,Fall,1180,2
6,QPASS 2024-2025-1 Summer.xlsm,2024-2025,Summer,1090,2
7,QPASS 2024-2025-2 Fall.xlsm,2024-2025,Fall,1344,3
8,QPASS 2024-2025-3 Winter.xlsm,2024-2025,Winter,1090,2
9,QPASS 2024-2025-4 Spring.xlsm,2024-2025,Spring,1400,2


Audit details remain in loader_audit: files, skipped_sheets, dropped_rows, non_numeric_values.


,facility_raw,category_raw,subtype_raw,source_file,source_row
0,Athena,Visitor,NaN,QPASS 2018-2019-1 Summer.xlsx,5
1,Athena,Visitor,NaN,QPASS 2018-2019-1 Summer.xlsx,6
2,Athena,A,NaN,QPASS 2018-2019-1 Summer.xlsx,7
3,Athena,B,NaN,QPASS 2018-2019-1 Summer.xlsx,8
4,Athena,Visitor,NaN,QPASS 2018-2019-1 Summer.xlsx,9


## 3. Inventory raw labels

Count each raw value, including missing values, and list its files, sheets, and academic-year/quarters. These are loader-record counts before time reshaping, not parking-space totals. Skipped and unsupported content remains covered by the shared schema audit.


In [3]:
# 3A — Inventory labels across all loaded workbook records
inventories = {}
for field in ("facility", "category", "subtype"):
    inventory = (
        records.groupby(f"{field}_raw", dropna=False, sort=False)
        .agg(
            count=("source_row", "size"),
            quarters=("quarter_label", lambda values: sorted(set(values.dropna()))),
            files=("source_file", lambda values: sorted(set(values.dropna()))),
            sheets=("source_sheet", lambda values: sorted(set(values.dropna()))),
        )
        .reset_index().sort_values("count", ascending=False)
    )
    inventories[field] = inventory
    assert inventory["count"].sum() == len(records)
    print(f"{field}: {len(inventory)} unique raw values including missing; showing up to 10")
    display(inventory.head(10))
# Full tables are available as inventories["facility"], ["category"], ["subtype"].


facility: 93 unique raw values including missing; showing up to 10


,facility_raw,count,quarters,files,sheets
4,Hopkins,1278,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
5,Pangea,1091,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
0,Athena,1063,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
3,Gilman,918,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
6,Torrey Pines Center North,734,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
2,Campus Point West,689,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
45,P602,663,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
72,South,638,"[2024-2025 Fall, 2024-2025 Spring, 2024-2025 S...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[Data Entry, QPASS Data Entry, QPASS Data Entr..."
7,Torrey Pines Center South,604,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
70,Scholars,564,"[2024-2025 Fall, 2024-2025 Spring, 2024-2025 S...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[Data Entry, QPASS Data Entry, QPASS Data Entr..."


category: 105 unique raw values including missing; showing up to 10


,category_raw,count,quarters,files,sheets
2,B,5748,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
1,A,4184,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
0,Visitor,3376,"[2018-2019 Summer, 2024-2025 Fall, 2024-2025 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2024-202...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
3,S,1999,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
95,Reserved,1076,"[2025-2026 Fall, 2025-2026 Spring, 2026-2027 F...","[QPASS 2025-2026-2 Fall.xlsm, QPASS 2025-2026-...","[Data Entry, QPASS Data Entry (Week 0), QPASS ..."
4,V,827,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
29,SR,590,"[2024-2025 Fall, 2024-2025 Spring, 2024-2025 S...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[Data Entry, QPASS Data Entry, QPASS Data Entr..."
38,Carpool,494,"[2024-2025 Spring, 2025-2026 Fall, 2025-2026 S...","[QPASS 2024-2025-4 Spring.xlsm, QPASS 2025-202...","[Data Entry, QPASS Data Entry (Week 0), QPASS ..."
16,SIO B,438,"[2019-2020 Fall, 2019-2020 Summer, 2024-2025 F...","[QPASS 2019-2020-1 Summer.xlsx, QPASS 2019-202...","[Fall19 Week 1 Data, Fall19 Week 2 Data, QPASS..."
9,Visitor: PAD,406,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ..."


subtype: 26 unique raw values including missing; showing up to 10


,subtype_raw,count,quarters,files,sheets
0,NaN,22600,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t..."
11,Scripps Institution of Oceanography,180,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
2,Gold Pool,168,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
9,Blue Pool,132,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
8,24/7 Enforced,36,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
15,B,36,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
17,Preuss,24,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
6,Patient/Visitor Reserved,24,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
10,2 Hours,24,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]
24,A,12,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry]


## 4. Check formatting

Trim whitespace, normalize word case and duration notation, and convert configured missing placeholders. Preserve raw columns and meaningful qualifiers.


### 4A — Preview basic formatting while preserving raw values

Creates `formatted_records` and `formatting_comparisons`. Trims whitespace, collapses repeated spaces, applies the provisional capitalization convention, and handles configured missing placeholders. Raw columns remain intact; meaningful restrictions remain part of the labels. Final label statuses are not assigned here.


In [4]:
# 4A — Apply current formatting rules, preserving the original columns
# Reload after editing src/labels.py so the notebook does not use cached code.
importlib.reload(label_tools)
format_label = label_tools.format_label
format_labels = label_tools.format_labels



# These placeholder conventions remain configurable; numeric zero is preserved.
formatted_records = format_labels(records)
comparison_columns = [
    "facility_raw", "facility", "category_raw", "category", "subtype_raw", "subtype"
]
display(formatted_records[comparison_columns].head(10))

# Compare unique raw/cleaned pairs, keeping restrictions and raw labels intact.
formatting_comparisons = {}
for field in ("facility", "category", "subtype"):
    pairs = formatted_records[[f"{field}_raw", field]].drop_duplicates()
    formatting_comparisons[field] = pairs

# Missing subtype alone does not establish whether the source field was absent.
print(f"Missing subtype values: {formatted_records['subtype'].isna().sum():,}")
print("Formatting preview only: final label_status requires alias/uncertainty review.")


,facility_raw,facility,category_raw,category,subtype_raw,subtype
0,Athena,Athena,Visitor,Visitor,NaN,<NA>
1,Athena,Athena,Visitor,Visitor,NaN,<NA>
2,Athena,Athena,A,A,NaN,<NA>
3,Athena,Athena,B,B,NaN,<NA>
4,Athena,Athena,Visitor,Visitor,NaN,<NA>
5,Athena,Athena,A,A,NaN,<NA>
6,Athena,Athena,B,B,NaN,<NA>
7,Athena,Athena,B,B,NaN,<NA>
8,Athena,Athena,B,B,NaN,<NA>
9,Athena,Athena,B,B,NaN,<NA>


Missing subtype values: 22,600
Formatting preview only: final label_status requires alias/uncertainty review.


### 4B — Review formatting changes in small pages

This summarizes formatting changes and provides a paged review table with raw labels, formatted labels, source-record counts, quarters, and files. Nothing here verifies aliases or assigns final statuses. Numeric zero remains a label requiring review; missing subtype alone is not evidence of an unresolved restriction.


In [5]:
# 4B — Summarize formatting changes and inspect labels without flooding output
formatting_review = {}
formatting_summary_rows = []
for field in ("facility", "category", "subtype"):
    raw = formatted_records[f"{field}_raw"]
    clean = formatted_records[field]
    same = (raw.astype("string").eq(clean.astype("string")).fillna(False)
            | (raw.isna() & clean.isna()))
    formatting_summary_rows.append({
        "field": field,
        "records": len(formatted_records),
        "changed_by_formatting": int((~same).sum()),
        "missing_after_formatting": int(clean.isna().sum()),
    })
    review = inventories[field].copy()
    formatter = label_tools.format_category if field == "category" else format_label
    review[field] = review[f"{field}_raw"].map(formatter)
    review["raw_repr"] = review[f"{field}_raw"].map(repr)
    review["clean_repr"] = review[field].map(repr)
    raw_text = review[f"{field}_raw"].astype("string")
    clean_text = review[field].astype("string")
    review["format_changed"] = ~(
        raw_text.eq(clean_text).fillna(False)
        | (review[f"{field}_raw"].isna() & review[field].isna())
    )
    formatting_review[field] = review

display(pd.DataFrame(formatting_summary_rows))


def review_labels(field="facility", changed_only=False, text=None, start=0, limit=10):
    """Show one small page of raw/formatted labels with counts and provenance."""
    if field not in formatting_review:
        raise ValueError("Choose facility, category, or subtype")
    selected = formatting_review[field]
    if changed_only:
        selected = selected.loc[selected["format_changed"]]
    if text is not None:
        selected = selected.loc[
            selected[f"{field}_raw"].astype("string").str.contains(
                text, case=False, regex=False, na=False
            )
        ]
    print(f"{field}: {len(selected)} matching labels; showing up to {limit} from offset {start}")
    display(selected.iloc[start:start + limit])


,field,records,changed_by_formatting,missing_after_formatting
0,facility,23428,32,0
1,category,23428,1663,63
2,subtype,23428,180,22600


In [6]:
review_labels("facility", changed_only=True)

facility: 1 matching labels; showing up to 10 from offset 0


,facility_raw,count,quarters,files,sheets,facility,raw_repr,clean_repr,format_changed
89,P602,32,"[2025-2026 Spring, 2026-2027 Fall, 2026-2027 S...","[QPASS 2025-2026-4 Spring.xlsm, QPASS 2026-202...",[Data Entry],P602,'P602 ','P602',True


In [7]:
review_labels("category", changed_only=True)

category: 51 matching labels; showing up to 10 from offset 0


,category_raw,count,quarters,files,sheets,category,raw_repr,clean_repr,format_changed
53,Visitor,220,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor,'Visitor ','Visitor',True
56,B,190,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",B,'B ','B',True
51,A,150,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",A,'A ','A',True
66,B Scripps Institution of Oceanography,110,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",B Scripps Institution Of Oceanography,'B Scripps Institution of Oceanography ','B Scripps Institution Of Oceanography',True
58,S,90,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",S,'S ','S',True
75,SR,70,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",SR,'SR ','SR',True
7,3hr A,69,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[3 Summer 2018 July 2nd, 4 Summer 2018 July 9t...",A 3 Hour,'3hr A','A 3 Hour',True
30,2 hour Visitor,64,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",Visitor 2 Hour,'2 hour Visitor','Visitor 2 Hour',True
35,2 hour A,42,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",A 2 Hour,'2 hour A','A 2 Hour',True
57,A 24/7 Enforced,40,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",A 24/7 Enforced,'A 24/7 Enforced ','A 24/7 Enforced',True


In [8]:
review_labels("subtype", changed_only=True)

subtype: 1 matching labels; showing up to 10 from offset 0


,subtype_raw,count,quarters,files,sheets,subtype,raw_repr,clean_repr,format_changed
11,Scripps Institution of Oceanography,180,[2026-2027 Fall],[QPASS 2026-2027-2 Fall.xlsm],[Data Entry],Scripps Institution Of Oceanography,'Scripps Institution of Oceanography','Scripps Institution Of Oceanography',True


### 4C — Complete category-formatting review (expand to inspect)

Run 4A and 4B first. This cell records every changed category, without truncating after ten, in a collapsible table. Raw and cleaned representations expose whitespace. Missing results are marked as needing a missing-value decision; other rows are formatting candidates, not verified semantic aliases. Counts may change after the duration-unit fix, so the table need not contain exactly 53 rows.

Save the notebook after running 4C so all rows are available for review. Quarters and file details remain in `formatting_review["category"]`; this table only omits those long lists from display.


In [9]:
# 4C — Save every changed category in one compact, collapsible review
changed_categories = formatting_review["category"].loc[
    formatting_review["category"]["format_changed"]
].copy()
changed_categories["review_note"] = changed_categories["category"].map(
    lambda value: "Review missing-value decision" if pd.isna(value)
    else "Formatting only; semantic alias not assessed"
)
category_review_all = changed_categories[
    ["raw_repr", "clean_repr", "count", "review_note"]
].reset_index(drop=True)
print(f"Complete category review: {len(category_review_all)} changed labels; no rows truncated.")
print(f"Affected source records: {category_review_all['count'].sum():,}")
display({
    "text/plain": category_review_all.to_string(index=False),
    "text/html": (
        "<details><summary>Expand all category formatting changes</summary>"
        + category_review_all.to_html(index=False, escape=True)
        + "</details>"
    ),
}, raw=True)


Complete category review: 51 changed labels; no rows truncated.
Affected source records: 1,663


raw_repr,clean_repr,count,review_note
'Visitor ','Visitor',220,Formatting only; semantic alias not assessed
'B ','B',190,Formatting only; semantic alias not assessed
'A ','A',150,Formatting only; semantic alias not assessed
'B Scripps Institution of Oceanography ','B Scripps Institution Of Oceanography',110,Formatting only; semantic alias not assessed
'S ','S',90,Formatting only; semantic alias not assessed
'SR ','SR',70,Formatting only; semantic alias not assessed
'3hr A','A 3 Hour',69,Formatting only; semantic alias not assessed
'2 hour Visitor','Visitor 2 Hour',64,Formatting only; semantic alias not assessed
'2 hour A','A 2 Hour',42,Formatting only; semantic alias not assessed
'A 24/7 Enforced ','A 24/7 Enforced',40,Formatting only; semantic alias not assessed


## 5. Review alias decisions

The JSON decision files contain supported aliases, restriction-preservation decisions, and unresolved wording. Whitespace changes stay in code. Evidence and source references are recorded in each decision’s notes.


### 5A — Review formatting groups before verifying aliases

Run after 4A. Creates `formatting_groups` and current `alias_reviews` tables. Groups show raw labels that converge after formatting; they do not establish semantic aliases. Verify meaningful aliases using source evidence, preserve restricted labels, and mark uncertainty unresolved before creating mapping files.


In [10]:
# 5A — Review saved decisions and formatting-only groups
mapping_columns = ["raw_value", "standard_value", "status", "notes"]
import json
alias_reviews = {field: pd.DataFrame(columns=mapping_columns)
                 for field in ("facility", "category", "subtype")}
alias_reviews["category"] = pd.DataFrame(
    json.loads((project_root / "config" / "category_aliases.json").read_text())
    + json.loads((project_root / "config" / "category_review_decisions.json").read_text())
)
alias_reviews["facility"] = pd.DataFrame(json.loads(
    (project_root / "config" / "facility_aliases.json").read_text()
))
print("Verified facility aliases:")
display(alias_reviews["facility"])
print("Current category decisions (scoped corrections precede unresolved fallbacks):")
display(alias_reviews["category"])

# Find multiple raw labels that converge under formatting. These are formatting
# groups, not verified semantic aliases. Keep all findings available in memory.
formatting_groups = {}
for field, pairs in formatting_comparisons.items():
    nonmissing = pairs.dropna(subset=[field])
    groups = nonmissing.groupby(field, sort=False).agg(
        raw_variants=(f"{field}_raw", lambda values: list(values))
    )
    formatting_groups[field] = groups[groups["raw_variants"].map(len).gt(1)].reset_index()
    print(f"{field}: {len(formatting_groups[field])} formatting groups")
    display(formatting_groups[field].head(5))

# Next: verify meaningful aliases against QPASS, Parking Inventory, or official
# UCSD sources. Record source references in notes. Keep restrictions separate.
# Evidence for each decision is recorded in its notes.


Verified facility aliases:


,raw_value,standard_value,status,notes
0,Gliderport,P386,verified,UCSD explicitly identifies Torrey Pines Glider...
1,P386 (Gliderport),P386,verified,UCSD event directions explicitly identify Park...
2,Osler,South,verified,UCSD identifies South Parking Structure as for...


Current category decisions (scoped corrections precede unresolved fallbacks):


,raw_value,standard_value,status,notes,scope
0,V,Visitor,verified,UCSD defines V as Visitor space. Exact categor...,NaN
1,A (24/7),A 24/7 Enforced,verified,Workbook screenshots show P402 Type A (24/7) i...,NaN
2,24/7 A,A 24/7 Enforced,verified,Exact category states both A and 24/7; canonic...,NaN
3,2 Person Carpool,Carpool 2 Person,verified,Workbook screenshots show Hopkins Type 2 Perso...,NaN
4,Visitor 5 Minute 5 minutes,Visitor 5 Minute,verified,Workbook review (2026-10-18): Summer 2025-2026...,NaN
5,Visitor 2 hour 2 hour,Visitor 2 Hour,verified,Workbook screenshots (2026-10-18): Summer 2025...,NaN
6,Carpool 2 Person Carpool,Carpool 2 Person,verified,Workbook review compared subsequent quarters (...,NaN
7,A Physician A,Physician A,verified,Workbook screenshots (2026-10-18): Summer 2025...,NaN
8,2hr As,A 2 Hour,verified,Workbook review compared the P602 two-hour A r...,{'facility_raw': 'P602'}
9,VA,VA,keep_separate,Provided QPASS screenshots show P782 as SS in ...,NaN


facility: 1 formatting groups


,facility,raw_variants
0,P602,"[P602, P602 ]"


category: 17 formatting groups


,category,raw_variants
0,Visitor,"[Visitor, Visitor ]"
1,A,"[A, A ]"
2,B,"[B, B ]"
3,S,"[S, S ]"
4,A 3 Hour,"[3hr A, A 3 hours]"


subtype: 0 formatting groups


,subtype,raw_variants


In [11]:
review_labels("category", text="Visitor", limit=30)
review_labels("category", text="V", limit=30)

category: 30 matching labels; showing up to 30 from offset 0


,category_raw,count,quarters,files,sheets,category,raw_repr,clean_repr,format_changed
0,Visitor,3376,"[2018-2019 Summer, 2024-2025 Fall, 2024-2025 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2024-202...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t...",Visitor,'Visitor','Visitor',False
9,Visitor: PAD,406,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: PAD,'Visitor: PAD','Visitor: PAD',False
8,Visitor: PBS,290,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: PBS,'Visitor: PBS','Visitor: PBS',False
53,Visitor,220,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor,'Visitor ','Visitor',True
10,Visitor: Meter,216,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: Meter,'Visitor: Meter','Visitor: Meter',False
30,2 hour Visitor,64,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",Visitor 2 Hour,'2 hour Visitor','Visitor 2 Hour',True
33,3 hour Visitor,32,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",Visitor 3 Hour,'3 hour Visitor','Visitor 3 Hour',True
77,Visitor 5 Minute 5 minutes,20,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor 5 Minute 5 Minutes,'Visitor 5 Minute 5 minutes','Visitor 5 Minute 5 Minutes',True
89,Visitor 3 hours,20,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor 3 Hour,'Visitor 3 hours','Visitor 3 Hour',True
52,Visitor Patient/Visitor Reserved 3 minutes,20,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor Patient/Visitor Reserved 3 Minutes,'Visitor Patient/Visitor Reserved 3 minutes','Visitor Patient/Visitor Reserved 3 Minutes',True


category: 40 matching labels; showing up to 30 from offset 0


,category_raw,count,quarters,files,sheets,category,raw_repr,clean_repr,format_changed
0,Visitor,3376,"[2018-2019 Summer, 2024-2025 Fall, 2024-2025 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2024-202...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t...",Visitor,'Visitor','Visitor',False
95,Reserved,1076,"[2025-2026 Fall, 2025-2026 Spring, 2026-2027 F...","[QPASS 2025-2026-2 Fall.xlsm, QPASS 2025-2026-...","[Data Entry, QPASS Data Entry (Week 0), QPASS ...",Reserved,'Reserved','Reserved',False
4,V,827,"[2018-2019 Fall, 2018-2019 Spring, 2018-2019 S...","[QPASS 2018-2019-1 Summer.xlsx, QPASS 2018-201...","[1 Summer 2018 June 18, 2 Summer 2018 June 25t...",V,'V','V',False
9,Visitor: PAD,406,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: PAD,'Visitor: PAD','Visitor: PAD',False
8,Visitor: PBS,290,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: PBS,'Visitor: PBS','Visitor: PBS',False
53,Visitor,220,[2025-2026 Summer],[QPASS 2025-2026-1 Summer.xlsm],"[QPASS Data Entry (Week 1), QPASS Data Entry (...",Visitor,'Visitor ','Visitor',True
10,Visitor: Meter,216,"[2018-2019 Fall, 2018-2019 Winter]","[QPASS 2018-2019-2 Fall.xlsx, QPASS 2018-2019-...","[F18 Oct1 Data, F18 Oct8, F18 Sep17 Data, F18 ...",Visitor: Meter,'Visitor: Meter','Visitor: Meter',False
97,Electric Vehicle,78,"[2025-2026 Fall, 2025-2026 Spring, 2026-2027 F...","[QPASS 2025-2026-2 Fall.xlsm, QPASS 2025-2026-...","[Data Entry, QPASS Data Entry (Week 0), QPASS ...",Electric Vehicle,'Electric Vehicle','Electric Vehicle',False
30,2 hour Visitor,64,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",Visitor 2 Hour,'2 hour Visitor','Visitor 2 Hour',True
33,3 hour Visitor,32,"[2024-2025 Fall, 2024-2025 Summer, 2024-2025 W...","[QPASS 2024-2025-1 Summer.xlsm, QPASS 2024-20...","[QPASS Data Entry, QPASS Data Entry (Week 0), ...",Visitor 3 Hour,'3 hour Visitor','Visitor 3 Hour',True


## 6. Apply decisions and review unresolved records

Apply formatting, exact mappings, and scoped source-row rules. Keep field statuses and derive `label_status` using this precedence: unresolved, keep_separate, standardized, unchanged. Missing subtype alone is allowed.


In [12]:
# 6A — Apply verified mappings and explicit unresolved-record flags
import importlib
import src.label_decisions as label_decisions
importlib.reload(label_decisions)

cleaned_records = label_decisions.standardize_labels(records)
working_records = cleaned_records
print("Statuses describe applied decisions; unchanged does not mean independently verified.")
display(cleaned_records["label_status"].value_counts().rename("records"))

# V means Visitor even when facility ownership remains uncertain.
v_rows = cleaned_records["category_raw"].map(format_label).eq("V").fillna(False)
display(cleaned_records.loc[v_rows, [
    "facility_raw", "facility", "category_raw", "category", "facility_status",
    "category_status", "label_status", "source_file", "source_row"
]].head(5))

unresolved_records = cleaned_records.loc[cleaned_records["label_status"].eq("unresolved")].copy()
print(f"Unresolved records: {len(unresolved_records):,} / {len(cleaned_records):,}")
display(unresolved_records[[
    "facility_raw", "facility", "category_raw", "category", "subtype_raw",
    "label_notes", "source_file", "sheet_name", "source_row"
]].head(10))
# All unresolved rows remain in unresolved_records; none are dropped.


Statuses describe applied decisions; unchanged does not mean independently verified.


label_status
unchanged        18276
keep_separate     2854
standardized      2110
unresolved         188
Name: records, dtype: int64

,facility_raw,facility,category_raw,category,facility_status,category_status,label_status,source_file,source_row
42,Pangea,Pangea,V,Visitor,unchanged,standardized,standardized,QPASS 2018-2019-1 Summer.xlsx,47
175,Pangea,Pangea,V,Visitor,unchanged,standardized,standardized,QPASS 2018-2019-1 Summer.xlsx,47
308,Pangea,Pangea,V,Visitor,unchanged,standardized,standardized,QPASS 2018-2019-1 Summer.xlsx,47
441,Pangea,Pangea,V,Visitor,unchanged,standardized,standardized,QPASS 2018-2019-1 Summer.xlsx,47
574,Pangea,Pangea,V,Visitor,unchanged,standardized,standardized,QPASS 2018-2019-1 Summer.xlsx,47


Unresolved records: 188 / 23,428


,facility_raw,facility,category_raw,category,subtype_raw,label_notes,source_file,sheet_name,source_row
1395,P017,P017,NaN,<NA>,NaN,category: missing or zero identifier;,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65
1400,P103,P103,2hr As,2hr As,NaN,category: Unverified meaning outside the expli...,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70
1527,P017,P017,NaN,<NA>,NaN,category: missing or zero identifier;,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65
1532,P103,P103,2hr As,2hr As,NaN,category: Unverified meaning outside the expli...,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70
1659,P017,P017,NaN,<NA>,NaN,category: missing or zero identifier;,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65
1664,P103,P103,2hr As,2hr As,NaN,category: Unverified meaning outside the expli...,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70
1791,P017,P017,NaN,<NA>,NaN,category: missing or zero identifier;,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65
1796,P103,P103,2hr As,2hr As,NaN,category: Unverified meaning outside the expli...,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70
1923,P017,P017,NaN,<NA>,NaN,category: missing or zero identifier;,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65
1928,P103,P103,2hr As,2hr As,NaN,category: Unverified meaning outside the expli...,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70


### 6B — Group unresolved records into source rows to investigate

Run after 6A. This does not fill or delete anything. Each group identifies the original workbook, sheet, and Excel row, plus the number of daily records affected. For example, ten unresolved P017 records may come from only two source rows. The displayed percentage counts each affected record once, even if several fields are unresolved.

Inspect the original `Type` cell and any merged-cell boundaries for missing categories. Leave unknown values unresolved unless source evidence supports a correction. Missing subtype by itself is allowed. The report only covers implemented decisions; additional semantic review is still required.


In [13]:
# 6B — Compact unresolved-source review (all details remain in label_reports)
from src.label_reports import build_label_reports, validate_label_output

label_reports = build_label_reports(cleaned_records)
unresolved_source_review = label_reports["unresolved_sources"]
print(f"{len(unresolved_records):,} unresolved records in {len(unresolved_source_review)} source-row groups.")
display(label_reports["status_summary"])

# Small columns and pages; no repeated daily rows or long source lists.
source_review_columns = [
    "source_file", "sheet_name", "source_row", "facility_raw", "category_raw",
    "subtype_raw", "affected_records", "label_notes",
]
display(unresolved_source_review[source_review_columns].head(15))


def review_unresolved(start=0, limit=10, file_contains=None):
    """Inspect one small page of unresolved source-row groups."""
    selected = unresolved_source_review
    if file_contains:
        selected = selected.loc[selected["source_file"].str.contains(
            file_contains, case=False, regex=False, na=False
        )]
    print(f"{len(selected)} matching source-row groups")
    display(selected[source_review_columns].iloc[start:start + limit])

# review_unresolved(start=15, limit=10)  # Further groups, if needed.
# review_unresolved(file_contains="2018-2019-1")


188 unresolved records in 82 source-row groups.


,label_status,records,percent
0,unchanged,18276,78.009220
1,standardized,2110,9.006317
2,keep_separate,2854,12.182004
3,unresolved,188,0.802459


,source_file,sheet_name,source_row,facility_raw,category_raw,subtype_raw,affected_records,label_notes
0,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,P017,NaN,NaN,5,category: missing or zero identifier;
1,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...
2,QPASS 2018-2019-1 Summer.xlsx,4 Summer 2018 July 9th,65,P017,NaN,NaN,5,category: missing or zero identifier;
3,QPASS 2018-2019-1 Summer.xlsx,4 Summer 2018 July 9th,70,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...
4,QPASS 2018-2019-2 Fall.xlsx,F18 Sep17 Data,73,P017,NaN,NaN,5,category: missing or zero identifier;
5,QPASS 2018-2019-2 Fall.xlsx,F18 Sep17 Data,78,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...
6,QPASS 2018-2019-2 Fall.xlsx,F18 Sept24 Data,79,P017,NaN,NaN,5,category: missing or zero identifier;
7,QPASS 2018-2019-2 Fall.xlsx,F18 Sept24 Data,84,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...
8,QPASS 2018-2019-2 Fall.xlsx,F18 Oct1 Data,80,P017,NaN,NaN,5,category: missing or zero identifier;
9,QPASS 2018-2019-2 Fall.xlsx,F18 Oct1 Data,85,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...


In [14]:
review_unresolved(start=15, limit=10)

82 matching source-row groups


,source_file,sheet_name,source_row,facility_raw,category_raw,subtype_raw,affected_records,label_notes
15,QPASS 2018-2019-3 Winter.xlsx,W19 W2 Data,86,P103,2hr As,NaN,5,category: Unverified meaning outside the expli...
16,QPASS 2018-2019-4 Spring.xlsx,S19 W1 Data,96,P017,NaN,NaN,5,category: missing or zero identifier;
17,QPASS 2018-2019-4 Spring.xlsx,S19 W2 Data,96,P017,NaN,NaN,5,category: missing or zero identifier;
18,QPASS 2019-2020-1 Summer.xlsx,SS19 W1 Data,103,P017,NaN,NaN,5,category: missing or zero identifier;
19,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,101,Pangea,V,NaN,5,category: UCSD defines V as Visitor space. Exa...
20,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,103,P017,NaN,NaN,5,category: missing or zero identifier;
21,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,20,P206,24/7,NaN,5,category: Permit type is absent; do not infer ...
22,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,120,Gliderport,$3,NaN,5,facility: UCSD explicitly identifies Torrey Pi...
23,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,124,P782,$3,NaN,5,category: Price alone does not establish a per...
24,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 2 Data,20,P206,24/7,NaN,5,category: Permit type is absent; do not infer ...


### 6C — Check restrictions and reviewed wording

Run 5A, 6A, and 6B first. Exact wording corrections preserve the original raw text and restrictions; populated subtype values remain separate fields. Missing categories remain unresolved.


In [15]:
# 6C — Review preserved visitor restrictions and unresolved wording
visitor_rows = cleaned_records["category"].astype("string").str.contains(
    r"\bvisitor\b", case=False, regex=True, na=False
)
restriction_review = (
    cleaned_records.loc[visitor_rows]
    .groupby(["category", "subtype", "category_status", "subtype_status", "label_status"],
             dropna=False, sort=False)
    .size().reset_index(name="records")
)
print(f"{len(restriction_review)} distinct visitor label/status combinations; showing up to 15.")
display(restriction_review.head(15))

wording_labels = pd.Series([
    entry["raw_value"] for entry in json.loads(
        (project_root / "config" / "category_review_decisions.json").read_text()
    )
]).map(format_label)
wording_review = cleaned_records.loc[cleaned_records["category_raw"].map(format_label).isin(wording_labels)]
print("Reviewed wording decisions (raw text preserved; restrictions retained):")
display(wording_review.groupby(
    ["category_raw", "category", "category_status", "label_status"], dropna=False
).size().reset_index(name="records"))


36 distinct visitor label/status combinations; showing up to 15.


,category,subtype,category_status,subtype_status,label_status,records
0,Visitor,NaN,unchanged,unchanged,unchanged,3208
1,Visitor,NaN,standardized,unchanged,standardized,1042
2,Visitor: PBS,NaN,keep_separate,unchanged,keep_separate,290
3,Visitor: PAD,NaN,keep_separate,unchanged,keep_separate,406
4,Visitor: Meter,NaN,keep_separate,unchanged,keep_separate,216
5,Visitor,NaN,standardized,unchanged,unresolved,5
6,Visitor 2 Hour,NaN,keep_separate,unchanged,keep_separate,114
7,Visitor 3 Hour,NaN,keep_separate,unchanged,keep_separate,52
8,Visitor 5 Min,NaN,keep_separate,unchanged,keep_separate,20
9,Visitor Police,NaN,keep_separate,unchanged,keep_separate,10


Reviewed wording decisions (raw text preserved; restrictions retained):


,category_raw,category,category_status,label_status,records
0,3.25,3.25,unresolved,unresolved,20
1,$3,$3,unresolved,unresolved,20
2,24/7,24/7,unresolved,unresolved,10
3,2hr As,2hr As,unresolved,unresolved,39
4,2hr As,A 2 Hour,keep_separate,keep_separate,69
5,A 2hr 12,A 2hr 12,unresolved,unresolved,10
6,A Physician A,Physician A,keep_separate,keep_separate,10
7,Carpool 2 Person Carpool,Carpool 2 Person,keep_separate,keep_separate,10
8,VA,VA,keep_separate,keep_separate,12
9,Visitor 2 hour 2 hour,Visitor 2 Hour,keep_separate,keep_separate,10


## 7. Validate and review

Check record counts, raw columns, provenance, repeatability, and report totals. Review complete cleaned-label inventories for additional aliases or uncertainty.


In [16]:
# Initial formatting checks using synthetic values; no workbook reads required.
assert format_label("  visitor   loading ") == "Visitor Loading"
assert format_label("V") == "V"
assert format_label("UCSD") == "UCSD"
assert pd.isna(format_label(" N/A "))
assert format_label("0") == "0"  # Preserve ambiguous labels for review.
assert format_label(format_label("  visitor   loading ")) == "Visitor Loading"
assert len(formatted_records) == len(records)
for field in ("facility", "category", "subtype"):
    pd.testing.assert_series_equal(
        records[f"{field}_raw"], formatted_records[f"{field}_raw"]
    )
print("Initial formatting and raw-value preservation checks passed.")


Initial formatting and raw-value preservation checks passed.


### 7B — Validate raw preservation and report totals

Run after 6B. These checks cover the loaded records across all files; they do not certify unreviewed aliases.


In [17]:
# 7B — Validate all loaded records without rereading any workbook
validate_label_output(records, cleaned_records)
rerun = label_decisions.standardize_labels(cleaned_records)
for column in ["facility", "category", "subtype", "label_status"]:
    pd.testing.assert_series_equal(cleaned_records[column], rerun[column])
assert label_reports["status_summary"]["records"].sum() == len(records)
assert label_reports["unresolved_sources"]["affected_records"].sum() == len(unresolved_records)
print(f"Integration checks passed across {records['source_file'].nunique()} workbooks.")
print("Raw labels/provenance preserved; counts reconcile; decisions are repeatable.")


Integration checks passed across 15 workbooks.
Raw labels/provenance preserved; counts reconcile; decisions are repeatable.


### 7C — Final review of every cleaned label

Run after 7B. Expand each table to inspect the complete inventory, including raw variants and field statuses; no labels are truncated. Full file and quarter lists remain available in `final_inventories`. This is a human review checkpoint, not an automatic approval. Look for overlooked abbreviations/aliases, inconsistent units, and restrictions that must remain distinct. Missing categories remain unresolved; missing subtype alone is allowed.

Save the notebook after running this cell so the entire list is available for review. We will update mappings only where evidence supports equivalence.


In [18]:
# 7C — Complete compact inventories, saved inside collapsible tables
import importlib
import src.label_reports as report_tools
importlib.reload(report_tools)
final_inventories = report_tools.build_cleaned_inventories(cleaned_records)

for field, inventory in final_inventories.items():
    assert inventory["records"].sum() == len(cleaned_records)
    compact = inventory[[field, "records", "raw_variants", "field_statuses"]]
    print(f"{field}: {len(compact)} values including missing; full table saved below.")
    display({
        "text/plain": compact.to_string(index=False),
        "text/html": (
            f"<details><summary>Expand ALL {field} labels ({len(compact)})</summary>"
            + compact.to_html(index=False, escape=True)
            + "</details>"
        ),
    }, raw=True)
# Provenance for a particular label remains available, e.g.:
# final_inventories["category"].loc[final_inventories["category"]["category"].eq("Visitor 2 Hour")]


facility: 90 values including missing; full table saved below.


facility,records,raw_variants,field_statuses
Athena,1063,['Athena'],[unchanged]
Campus Point East,423,['Campus Point East'],[unchanged]
Campus Point West,689,['Campus Point West'],[unchanged]
Gilman,918,['Gilman'],[unchanged]
Hopkins,1278,['Hopkins'],[unchanged]
Pangea,1086,['Pangea'],[unchanged]
Torrey Pines Center North,734,['Torrey Pines Center North'],[unchanged]
Torrey Pines Center South,604,['Torrey Pines Center South'],[unchanged]
P011,175,['P011'],[unchanged]
P012,175,['P012'],[unchanged]


category: 77 values including missing; full table saved below.


category,records,raw_variants,field_statuses
Visitor,4423,"['V', 'Visitor ', 'Visitor']","[standardized, unchanged]"
A,4334,"['A ', 'A']","[standardized, unchanged]"
B,5938,"['B ', 'B']","[standardized, unchanged]"
S,2089,"['S ', 'S']","[standardized, unchanged]"
NaN,63,[nan],[unresolved]
2hr As,39,['2hr As'],[unresolved]
A 3 Hour,79,"['3hr A', 'A 3 hours']",[keep_separate]
A 2 Hour,141,"['2 hour A', '2hr A', '2hr As', 'A 2 Hour', 'A 2 Hour']",[keep_separate]
Visitor: PBS,290,['Visitor: PBS'],[keep_separate]
Visitor: PAD,406,['Visitor: PAD'],[keep_separate]


subtype: 26 values including missing; full table saved below.


subtype,records,raw_variants,field_statuses
NaN,22600,[nan],[unchanged]
A/B,12,['A/B'],[keep_separate]
Gold Pool,168,['Gold Pool'],[keep_separate]
Loading,12,['Loading'],[keep_separate]
Loading Reserved,12,['Loading Reserved'],[keep_separate]
Infusion,12,['Infusion'],[keep_separate]
Patient/Visitor Reserved,24,['Patient/Visitor Reserved'],[keep_separate]
Radiation,12,['Radiation'],[keep_separate]
24/7 Enforced,36,['24/7 Enforced'],[keep_separate]
Blue Pool,132,['Blue Pool'],[keep_separate]


In [19]:
remaining = cleaned_records["category"].eq("2hr As").fillna(False)

display(
    cleaned_records.loc[remaining]
    .groupby(
        ["facility_raw", "source_file", "sheet_name", "source_row"],
        dropna=False,
    )
    .size()
    .reset_index(name="records")
)

,facility_raw,source_file,sheet_name,source_row,records
0,P103,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70,5
1,P103,QPASS 2018-2019-1 Summer.xlsx,4 Summer 2018 July 9th,70,5
2,P103,QPASS 2018-2019-2 Fall.xlsx,F18 Oct1 Data,85,5
3,P103,QPASS 2018-2019-2 Fall.xlsx,F18 Oct8,85,4
4,P103,QPASS 2018-2019-2 Fall.xlsx,F18 Sep17 Data,78,5
5,P103,QPASS 2018-2019-2 Fall.xlsx,F18 Sept24 Data,84,5
6,P103,QPASS 2018-2019-3 Winter.xlsx,W19 W1 Data,21,5
7,P103,QPASS 2018-2019-3 Winter.xlsx,W19 W2 Data,86,5


### 7D — Locate uncertain labels in the original workbooks

Use exact label matches to list workbook/sheet/Excel-row provenance. Repeated daily observations are grouped for inspection, not deleted. An acronym found on the web is not enough to identify a QPASS category; VA is not automatically mapped to the medical center. `Visitor 2-Hour Limit 6` remains intact; the final number is not silently removed.


In [20]:
# 7D — Trace a label to its original source rows

def locate_label(value, field="category", start=0, limit=15):
    """Find source rows for one exact formatted raw label, with daily counts."""
    if field not in {"facility", "category", "subtype"}:
        raise ValueError("Choose facility, category, or subtype")
    mask = records[f"{field}_raw"].map(format_label).eq(format_label(value)).fillna(False)
    selected = records.loc[mask]
    trace = (
        selected.groupby(["source_file", "sheet_name", "source_row", "facility_raw",
                          "category_raw", "subtype_raw"], dropna=False, sort=False)
        .size().reset_index(name="daily_records")
    )
    print(f"{value!r}: {len(selected)} records, {len(trace)} source rows; showing up to {limit}")
    display(trace.iloc[start:start + limit])
    return trace

uncertain_a_sources = locate_label("2hr As")


'2hr As': 108 records, 22 source rows; showing up to 15


,source_file,sheet_name,source_row,facility_raw,category_raw,subtype_raw,daily_records
0,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,70,P103,2hr As,NaN,5
1,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,116,P602,2hr As,NaN,5
2,QPASS 2018-2019-1 Summer.xlsx,4 Summer 2018 July 9th,70,P103,2hr As,NaN,5
3,QPASS 2018-2019-1 Summer.xlsx,4 Summer 2018 July 9th,116,P602,2hr As,NaN,5
4,QPASS 2018-2019-2 Fall.xlsx,F18 Sep17 Data,78,P103,2hr As,NaN,5
5,QPASS 2018-2019-2 Fall.xlsx,F18 Sep17 Data,123,P602,2hr As,NaN,5
6,QPASS 2018-2019-2 Fall.xlsx,F18 Sept24 Data,84,P103,2hr As,NaN,5
7,QPASS 2018-2019-2 Fall.xlsx,F18 Sept24 Data,129,P602,2hr As,NaN,5
8,QPASS 2018-2019-2 Fall.xlsx,F18 Oct1 Data,85,P103,2hr As,NaN,5
9,QPASS 2018-2019-2 Fall.xlsx,F18 Oct1 Data,130,P602,2hr As,NaN,5


In [21]:
va_sources = locate_label("VA")

'VA': 12 records, 12 source rows; showing up to 15


,source_file,sheet_name,source_row,facility_raw,category_raw,subtype_raw,daily_records
0,QPASS 2026-2027-2 Fall.xlsm,Data Entry,126,P782,VA,NaN,1
1,QPASS 2026-2027-2 Fall.xlsm,Data Entry,304,P782,VA,NaN,1
2,QPASS 2026-2027-2 Fall.xlsm,Data Entry,482,P782,VA,NaN,1
3,QPASS 2026-2027-2 Fall.xlsm,Data Entry,660,P782,VA,NaN,1
4,QPASS 2026-2027-2 Fall.xlsm,Data Entry,838,P782,VA,NaN,1
5,QPASS 2026-2027-2 Fall.xlsm,Data Entry,1016,P782,VA,NaN,1
6,QPASS 2026-2027-2 Fall.xlsm,Data Entry,1194,P782,VA,NaN,1
7,QPASS 2026-2027-2 Fall.xlsm,Data Entry,1372,P782,VA,NaN,1
8,QPASS 2026-2027-2 Fall.xlsm,Data Entry,1550,P782,VA,NaN,1
9,QPASS 2026-2027-2 Fall.xlsm,Data Entry,1728,P782,VA,NaN,1


## 8. Export results

Save inventories, unresolved values and records, file/quarter summaries, and raw/cleaned examples. The full cleaned dataset stays local; compact reports can accompany the PR.


### Integration with Issue #2

These counts use loader records before time reshaping. Issue #2 can apply the same functions before or after reshaping if raw labels and provenance are retained. Confirm the counting unit and duplicate-check fields during integration.


In [22]:
# 8A — Summary for the selected source workbooks
print(f"Workbooks in analysis: {records['source_file'].nunique()}")
print(f"Records: {len(cleaned_records):,}")
display(label_reports["status_summary"])


Workbooks in analysis: 15
Records: 23,428


,label_status,records,percent
0,unchanged,18276,78.009220
1,standardized,2110,9.006317
2,keep_separate,2854,12.182004
3,unresolved,188,0.802459


### Data scope

QPASS surveys cover the beginning of selected quarters, with gaps in the archive. Some quarters include more than two weeks. The notebook follows the shared loader’s source-sheet rules and the team’s Winter 2025–2026 exclusion. These records describe historical surveys, not live parking availability.


### 8B — Write the deliverables

Run after validation. This writes compact review artifacts and a full local cleaned dataset under `results/issue3/`. The export checks raw-value preservation before writing.


In [23]:
# 8B — Write inventories, examples, unresolved reports, and cleaned records
importlib.reload(report_tools)
report_dir = project_root / "results" / "issue3"
written = report_tools.export_label_reports(
    records, cleaned_records, inventories, report_dir
)
print(f"Saved {len(written)} files to {report_dir}")
display(pd.DataFrame({"report": [path.name for path in written]}))


Saved 14 files to /Users/daira/parksmart/results/issue3


,report
0,status_summary.json
1,unresolved_sources.json
2,unresolved_values.json
3,by_file_quarter.json
4,unresolved_records.json
5,facility_inventory.json
6,category_inventory.json
7,subtype_inventory.json
8,facility_cleaned_inventory.json
9,category_cleaned_inventory.json
